[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part3_spatial_ai/ch14_differentiable_rendering/01_nerf_and_gaussian_splatting.ipynb)

# 第14章: 微分可能レンダリング - NeRF と 3D Gaussian Splatting

本ノートブックでは、微分可能レンダリングの2大手法について学びます。

## 学習内容
1. **ボリュームレンダリング方程式**: 光線に沿った色と密度の積分
2. **Tiny 1D NeRF**: MLPによるレイ上の色予測と体積レンダリング
3. **2D Gaussian Splatting**: ガウシアンのアルファブレンディングによるレンダリング
4. **微分可能性**: なぜこれらの表現がSLAMに有用か

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.colors import to_rgba
np.random.seed(42)

## 1. ボリュームレンダリング方程式

NeRFの核となるボリュームレンダリングの式：

$$C(\mathbf{r}) = \int_{t_n}^{t_f} T(t) \cdot \sigma(\mathbf{r}(t)) \cdot \mathbf{c}(\mathbf{r}(t), \mathbf{d}) \, dt$$

ここで：
- $T(t) = \exp\left(-\int_{t_n}^{t} \sigma(\mathbf{r}(s)) ds\right)$ は透過率（transmittance）
- $\sigma$ は密度（density）、$\mathbf{c}$ は色（color）
- 離散化すると：$C = \sum_{i=1}^{N} T_i \cdot \alpha_i \cdot c_i$, $\alpha_i = 1 - \exp(-\sigma_i \delta_i)$

In [ ]:
# --- Volume Rendering Equation Demo ---

def volume_rendering(colors, densities, deltas):
    """
    Discrete volume rendering along a ray.
    colors: (N, 3) RGB colors at sample points
    densities: (N,) density at each sample
    deltas: (N,) distance between consecutive samples
    Returns: rendered color (3,), weights (N,), transmittance (N,)
    """
    alpha = 1.0 - np.exp(-densities * deltas)  # opacity
    # Transmittance: cumulative product of (1 - alpha) for all previous samples
    transmittance = np.ones(len(alpha))
    for i in range(1, len(alpha)):
        transmittance[i] = transmittance[i-1] * (1.0 - alpha[i-1])
    
    weights = transmittance * alpha
    rendered_color = np.sum(weights[:, None] * colors, axis=0)
    return rendered_color, weights, transmittance

# Create a synthetic 1D scene along a ray
N_samples = 100
t = np.linspace(0, 10, N_samples)  # distance along ray
deltas = np.diff(t, prepend=t[0])

# Scene: two "objects" (density peaks) with different colors
densities = np.zeros(N_samples)
colors = np.zeros((N_samples, 3))

# Object 1: red ball at t=3
mask1 = np.exp(-0.5 * ((t - 3.0) / 0.5)**2) * 8.0
densities += mask1
colors += mask1[:, None] * np.array([1.0, 0.2, 0.1])

# Object 2: blue ball at t=7
mask2 = np.exp(-0.5 * ((t - 7.0) / 0.8)**2) * 5.0
densities += mask2
colors += mask2[:, None] * np.array([0.1, 0.3, 1.0])

# Normalize colors
color_norm = np.maximum(densities, 1e-8)
colors = colors / color_norm[:, None]
colors = np.clip(colors, 0, 1)

rendered_color, weights, transmittance = volume_rendering(colors, densities, deltas)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].fill_between(t, densities, alpha=0.3, color='purple')
axes[0, 0].plot(t, densities, color='purple', linewidth=2)
axes[0, 0].set_title('Density $\\sigma(t)$ along ray', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('t (distance along ray)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].grid(True, alpha=0.3)

# Show colors as colored bars
for i in range(N_samples - 1):
    axes[0, 1].axvspan(t[i], t[i+1], color=colors[i], alpha=0.8)
axes[0, 1].set_title('Color $c(t)$ along ray', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('t (distance along ray)')
axes[0, 1].set_xlim(t[0], t[-1])

axes[1, 0].plot(t, transmittance, color='green', linewidth=2, label='T(t)')
axes[1, 0].fill_between(t, weights, alpha=0.4, color='orange', label='weights')
axes[1, 0].plot(t, weights, color='orange', linewidth=2)
axes[1, 0].set_title('Transmittance & Weights', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('t (distance along ray)')
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

# Final rendered color
axes[1, 1].add_patch(plt.Rectangle((0.2, 0.2), 0.6, 0.6, color=rendered_color))
axes[1, 1].set_title(f'Rendered Pixel Color\nRGB=({rendered_color[0]:.2f}, {rendered_color[1]:.2f}, {rendered_color[2]:.2f})',
                      fontsize=13, fontweight='bold')
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_aspect('equal')

plt.tight_layout()
plt.savefig('volume_rendering.png', dpi=100, bbox_inches='tight')
plt.show()
print("前方の赤い物体が手前にあるため、レンダリング色は赤が支配的です")

## 2. Tiny 1D NeRF - MLPによるシーン表現の学習

NeRFでは、MLP（多層パーセプトロン）がシーンの暗黙的表現として機能します。入力は3D座標、出力は色と密度です。

ここでは、1Dの単純なケースで「MLPがレイ上の色分布を学習する」過程を実装します。numpy のみで簡易的なMLPと勾配降下法を実装します。

In [ ]:
# --- Tiny 1D NeRF: MLP learns to represent a 1D scene ---

class TinyMLP:
    """Simple 2-layer MLP: input -> hidden -> output, with ReLU activation."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        # He initialization
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros(output_dim)
    
    def forward(self, x):
        """Forward pass, storing intermediates for backprop."""
        self.x = x
        self.z1 = x @ self.W1 + self.b1
        self.a1 = np.maximum(0, self.z1)  # ReLU
        self.z2 = self.a1 @ self.W2 + self.b2
        # Sigmoid output for color (0-1), softplus for density (>0)
        return self.z2
    
    def backward(self, grad_output, lr=0.001):
        """Simple backpropagation with SGD."""
        # Gradient through output layer
        grad_W2 = self.a1.T @ grad_output
        grad_b2 = np.sum(grad_output, axis=0)
        grad_a1 = grad_output @ self.W2.T
        
        # Gradient through ReLU
        grad_z1 = grad_a1 * (self.z1 > 0).astype(float)
        
        # Gradient through input layer
        grad_W1 = self.x.T @ grad_z1
        grad_b1 = np.sum(grad_z1, axis=0)
        
        # SGD update
        self.W1 -= lr * grad_W1
        self.b1 -= lr * grad_b1
        self.W2 -= lr * grad_W2
        self.b2 -= lr * grad_b2

def positional_encoding(t, L=6):
    """Positional encoding: sin/cos features for better high-freq learning."""
    features = [t]
    for freq in range(L):
        features.append(np.sin(2**freq * np.pi * t))
        features.append(np.cos(2**freq * np.pi * t))
    return np.column_stack(features)

# Target 1D scene: color as function of position along ray
t_samples = np.linspace(0, 1, 64).reshape(-1, 1)
# Target: two color peaks (red at 0.3, blue at 0.7)
target_color = (np.exp(-((t_samples - 0.3) / 0.08)**2) * 0.9 + 
                np.exp(-((t_samples - 0.7) / 0.1)**2) * 0.6)

# Encode input with positional encoding
t_encoded = positional_encoding(t_samples, L=4)
input_dim = t_encoded.shape[1]

# Train MLP to predict color from position
mlp = TinyMLP(input_dim, 32, 1)
losses = []

for epoch in range(2000):
    # Forward
    pred = mlp.forward(t_encoded)
    
    # MSE loss
    loss = np.mean((pred - target_color)**2)
    losses.append(loss)
    
    # Backward
    grad = 2.0 * (pred - target_color) / len(pred)
    mlp.backward(grad, lr=0.005)

# Plot training results
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].semilogy(losses)
axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_samples, target_color, 'b-', linewidth=2, label='Target')
axes[1].plot(t_samples, pred, 'r--', linewidth=2, label='MLP Prediction')
axes[1].set_title('1D NeRF: Color along Ray', fontsize=13, fontweight='bold')
axes[1].set_xlabel('t (position along ray)')
axes[1].set_ylabel('Color intensity')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# Without positional encoding
mlp_no_pe = TinyMLP(1, 32, 1)
for epoch in range(2000):
    pred_no_pe = mlp_no_pe.forward(t_samples)
    grad = 2.0 * (pred_no_pe - target_color) / len(pred_no_pe)
    mlp_no_pe.backward(grad, lr=0.005)

axes[2].plot(t_samples, target_color, 'b-', linewidth=2, label='Target')
axes[2].plot(t_samples, mlp_no_pe.forward(t_samples), 'r--', linewidth=2, label='Without PE')
axes[2].plot(t_samples, mlp.forward(t_encoded), 'g--', linewidth=2, label='With PE')
axes[2].set_title('Effect of Positional Encoding', fontsize=13, fontweight='bold')
axes[2].set_xlabel('t (position along ray)')
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tiny_nerf_1d.png', dpi=100, bbox_inches='tight')
plt.show()
print("位置エンコーディング(PE)により、MLPが高周波の詳細を学習できるようになります")

## 3. 2D Gaussian Splatting

3D Gaussian Splatting (3DGS) は、シーンを多数の3Dガウシアンで表現し、それらを画像平面に「スプラット」（投影）してレンダリングします。

各ガウシアンは以下のパラメータを持ちます：
- **位置** $\mu$: 中心座標
- **共分散** $\Sigma$: 形状と向き
- **色** $c$: RGB色
- **不透明度** $\alpha$: 透過度

レンダリングは前後順でアルファブレンディング：
$$C = \sum_{i=1}^{N} c_i \cdot \alpha_i \cdot \prod_{j=1}^{i-1}(1 - \alpha_j)$$

In [ ]:
# --- 2D Gaussian Splatting Visualization ---

def gaussian_2d(x, y, mu, cov):
    """Evaluate 2D Gaussian at positions (x, y)."""
    dx = np.stack([x - mu[0], y - mu[1]], axis=-1)  # (..., 2)
    cov_inv = np.linalg.inv(cov)
    # Mahalanobis distance
    maha = np.einsum('...i,ij,...j', dx, cov_inv, dx)
    return np.exp(-0.5 * maha)

def render_gaussians(gaussians, img_size=(200, 200)):
    """Render 2D Gaussians with alpha blending (front-to-back)."""
    H, W = img_size
    x = np.linspace(0, 10, W)
    y = np.linspace(0, 10, H)
    xx, yy = np.meshgrid(x, y)
    
    # Sort by depth (here we use a 'depth' attribute, front first)
    sorted_g = sorted(gaussians, key=lambda g: g['depth'])
    
    rendered = np.zeros((H, W, 3))
    accumulated_alpha = np.zeros((H, W))
    
    for g in sorted_g:
        gauss_val = gaussian_2d(xx, yy, g['mu'], g['cov'])
        alpha = gauss_val * g['opacity']
        alpha = np.clip(alpha, 0, 1)
        
        # Alpha blending: C += alpha_i * (1 - accumulated) * color_i
        contrib = alpha * (1 - accumulated_alpha)
        for c in range(3):
            rendered[:, :, c] += contrib * g['color'][c]
        accumulated_alpha += contrib
    
    return np.clip(rendered, 0, 1), accumulated_alpha

# Define some 2D Gaussians (simulating projected 3D Gaussians)
gaussians = [
    {'mu': [3, 4], 'cov': [[0.8, 0.2], [0.2, 0.5]], 'color': [1.0, 0.2, 0.1], 'opacity': 0.9, 'depth': 1},
    {'mu': [6, 6], 'cov': [[1.2, -0.3], [-0.3, 0.8]], 'color': [0.1, 0.8, 0.2], 'opacity': 0.8, 'depth': 2},
    {'mu': [5, 3], 'cov': [[0.5, 0.0], [0.0, 1.5]], 'color': [0.2, 0.3, 1.0], 'opacity': 0.85, 'depth': 3},
    {'mu': [7, 2], 'cov': [[0.6, 0.1], [0.1, 0.4]], 'color': [0.9, 0.9, 0.1], 'opacity': 0.7, 'depth': 4},
    {'mu': [2, 7], 'cov': [[1.0, 0.5], [0.5, 1.0]], 'color': [0.8, 0.1, 0.8], 'opacity': 0.75, 'depth': 5},
]

rendered_img, alpha_map = render_gaussians(gaussians)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Show individual Gaussians
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
colors_list = ['red', 'green', 'blue', 'yellow', 'purple']
for i, g in enumerate(gaussians):
    # Draw ellipse from covariance
    eigvals, eigvecs = np.linalg.eigh(g['cov'])
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    for scale in [1, 2]:
        ell = Ellipse(g['mu'], 2*scale*np.sqrt(eigvals[0]), 2*scale*np.sqrt(eigvals[1]),
                      angle=angle, fill=False, edgecolor=colors_list[i], linewidth=2, alpha=0.5+0.3/scale)
        ax.add_patch(ell)
    ax.plot(g['mu'][0], g['mu'][1], 'o', color=colors_list[i], markersize=8)
ax.set_title('Individual 2D Gaussians', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Rendered image
axes[1].imshow(rendered_img, origin='lower', extent=[0, 10, 0, 10])
axes[1].set_title('Alpha-Blended Rendering', fontsize=13, fontweight='bold')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')

# Alpha accumulation
im = axes[2].imshow(alpha_map, origin='lower', extent=[0, 10, 0, 10], cmap='gray')
axes[2].set_title('Accumulated Opacity', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.savefig('gaussian_splatting_2d.png', dpi=100, bbox_inches='tight')
plt.show()
print("各ガウシアンのアルファブレンディングにより、半透明な重なりが表現されています")

## 演習問題

1. **ボリュームレンダリング**: 物体の密度を変えて（例：前方の物体を半透明にして）、レンダリング結果がどう変わるか確認してください。
2. **位置エンコーディングの周波数**: `L` パラメータを変えて、MLPの表現力への影響を調べてください。
3. **ガウシアンの数**: ガウシアンの数を増やし（20個以上）、より複雑なシーンを表現してみてください。

## まとめ

- **ボリュームレンダリング**はNeRFの基盤であり、レイに沿った色と密度の積分でピクセル色を計算する
- **NeRF**はMLPでシーンの暗黙的表現を学習し、**位置エンコーディング**が高周波詳細の表現に不可欠
- **3D Gaussian Splatting**は明示的なガウシアンプリミティブを使い、アルファブレンディングで高速レンダリングを実現
- 両手法とも**微分可能**であり、画像からの逆最適化でシーンパラメータを推定できるため、SLAMとの統合が可能